In [1]:
import os, numpy as np, requests, torch
from types import SimpleNamespace
from dotenv import load_dotenv
from transformers import GPT2TokenizerFast

# context_cite 0.0.4 uses Styler.applymap which was renamed to .map in pandas 2.x
from pandas.io.formats.style import Styler
if not hasattr(Styler, "applymap"):
    Styler.applymap = Styler.map

from context_cite import ContextCiter

load_dotenv(dotenv_path="../.env")

# LLM endpoint — used for prompt_logprobs calls during ContextCite ablation
# Requires a vllm-compatible endpoint (prompt_logprobs is a vllm extension,
# not supported by the standard OpenAI API)
API_KEY  = os.getenv("OPENAI_API_KEY")
LLM_URL = os.getenv("OPENAI_BASE_URL").rstrip("/") + "/v1/chat/completions"
MODEL    = os.getenv("LLM_MODEL", "local-model-mini")

# Backend RAG pipeline — used to get answer + retrieved chunks
BACKEND_URL   = os.getenv("BACKEND_API_URL")
WORKSPACE_ID  = os.getenv("BACKEND_WORKSPACE_ID")
BACKEND_TOKEN = os.getenv("BACKEND_API_KEY")

print(f"LLM: {MODEL} @ {LLM_URL}")

LLM: local-model-mini @ https://chat.zingplay.com/api/v1/chat/completions


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lap15114/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# GPT-2 is used as a structural tokenizer shell — it gives ContextCiter a
# PreTrainedTokenizerBase with pad/eos tokens, encode/decode, token_to_chars,
# and the full .pad() method DataCollatorForSeq2Seq needs.
# The chat_template matches the ChatML format the LLM endpoint expects.
# Actual log-probabilities come from the API, not GPT-2 weights.
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

tokenizer.chat_template = (
    "{% for message in messages %}"
    "<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)

In [3]:
import json

QUERY = "Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?"

try:
    backend_resp = requests.post(
        f"{BACKEND_URL}/api/v1/workspaces/{WORKSPACE_ID}/chat",
        headers={"Authorization": f"Bearer {BACKEND_TOKEN}"},
        json={"query": QUERY, "mode": "agent", "top_k": 2},
        timeout=120,
    )
    backend_resp.raise_for_status()
    data = backend_resp.json()
    print("Loaded from live backend API")
except Exception as e:
    print(f"Backend unreachable ({e.__class__.__name__}), loading from chat_output.txt")
    with open("chat_output.txt") as f:
        data = json.load(f)

RESPONSE = data["answer"]
CONTEXT  = "\n\n".join(chunk["content"] for chunk in data["knowledge_sources"])

print(f"Knowledge sources: {len(data['knowledge_sources'])}")
print(f"\nRESPONSE:\n{RESPONSE}")
print(f"\nCONTEXT:\n{CONTEXT}")

Loaded from live backend API
Knowledge sources: 5

RESPONSE:
Dựa trên các tài liệu hiện có, số ngày nghỉ phép năm cụ thể cho mỗi nhân viên VNG **không được nêu rõ** trong các đoạn trích đã tìm thấy.

Tuy nhiên, các tài liệu cung cấp các quy định liên quan đến việc sử dụng và bảo lưu phép năm:
*   Nhân viên có thể đăng ký nghỉ tối đa bằng số ngày phép năm còn lại cộng thêm tối đa **05 ngày** phép năm tạm ứng của năm hiện tại.
*   Số ngày phép năm chưa sử dụng hết có thể được bảo lưu sang năm tiếp theo (tối đa **05 ngày**) và phải được sử dụng không muộn hơn ngày **31/03** của năm sau.
*   Các ngày phép bảo lưu không sử dụng hết sau thời hạn trên sẽ bị hủy bỏ và không được chuyển đổi thành tiền.

Để biết chính xác số ngày phép năm được cấp cho từng nhân viên (thường phụ thuộc vào thâm niên), bạn cần tham khảo thêm các phần khác trong chính sách hoặc liên hệ bộ phận Nhân sự.

CONTEXT:
### 3.1.2. Quy định cách sử dụng

- Số ngày nghỉ tối đa Nhân viên có thể đăng ký nghỉ = Số ngày còn lại c

In [14]:
# --- API helpers ---------------------------------------------------------

def _actual_tokens(raw_logprobs):
    """
    Return [(decoded_text, logprob)] for the actual input token at each position.

    With prompt_logprobs=1 the API returns, per position:
      1 entry  → that token IS the top-1 prediction (actual = top-1)
      2 entries → actual token was NOT top-1; it is the non-rank-1 entry
    Position 0 is always None (no conditioning).
    """
    result = []
    for entry in raw_logprobs:
        if entry is None:
            result.append(("<|im_start|>", None))
            continue
        if len(entry) == 1:
            v = list(entry.values())[0]
        else:
            v = max(entry.values(), key=lambda x: x["rank"])
        result.append((v["decoded_token"], v["logprob"]))
    return result


def _extract_user_content(prompt_text: str) -> str:
    """Pull the user-turn text out of a ChatML-formatted string."""
    start = prompt_text.find("<|im_start|>user\n")
    if start == -1:
        raise ValueError(f"No user turn found in prompt: {prompt_text[:80]!r}")
    start += len("<|im_start|>user\n")
    end = prompt_text.find("<|im_end|>", start)
    return prompt_text[start:] if end == -1 else prompt_text[start:end]


def _api_response_token_logprobs(user_content: str, response_text: str):
    """
    Call the LLM endpoint with prompt_logprobs=1 and return a list of
    (decoded_token, logprob) for every token that belongs to response_text.

    The model may prepend <think>...</think> to the assistant turn, so we
    locate response_text by rfind rather than a fixed offset.
    """
    print("USER CONTENT:", user_content[:200])
    print("RESPONSE TEXT:", response_text[:200])
    r = requests.post(
        LLM_URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": MODEL,
            "messages": [
                {"role": "user",      "content": user_content},
                {"role": "assistant", "content": response_text},
            ],
            "temperature": 0,
            "max_tokens": 1,
            "prompt_logprobs": 1,
        },
    )
    if not r.ok:
        print("STATUS:", r.status_code)
        print("BODY:", r.text[:500])
    r.raise_for_status()

    tokens   = _actual_tokens(r.json()["prompt_logprobs"])
    full_txt = "".join(t for t, _ in tokens)

    start = full_txt.rfind(response_text)
    if start == -1:
        start = full_txt.rfind(response_text.strip())
    if start == -1:
        raise ValueError(
            f"response_text not found in decoded sequence.\nTail: {full_txt[-300:]!r}"
        )
    end = start + len(response_text)

    out, pos = [], 0
    for decoded, lp in tokens:
        tok_end = pos + len(decoded)
        if lp is not None and pos >= start and tok_end <= end:
            out.append((decoded, lp))
        pos = tok_end
        if pos >= end:
            break
    return out


def _align_to_gpt2_tokens(api_tokens, response_text: str, gpt2_response_ids: list):
    """
    Map per-API-token logprobs onto per-GPT2-token logprobs using character spans.

    For each GPT-2 token we compute a weighted average of the API logprobs whose
    character spans overlap with it (weighted by overlap length).
    """
    n = len(gpt2_response_ids)
    if not api_tokens:
        return [-5.0] * n

    api_spans, pos = [], 0
    for decoded, lp in api_tokens:
        api_spans.append((pos, pos + len(decoded), lp))
        pos += len(decoded)

    enc = tokenizer(response_text, add_special_tokens=False)

    result = []
    for j in range(n):
        cs = enc.token_to_chars(j)
        our_s, our_e = cs.start, cs.end

        overlapping = [
            (lp, min(our_e, a_e) - max(our_s, a_s))
            for a_s, a_e, lp in api_spans
            if min(our_e, a_e) > max(our_s, a_s)
        ]
        if overlapping:
            total = sum(o for _, o in overlapping)
            result.append(sum(lp * o / total for lp, o in overlapping))
        else:
            mid = (our_s + our_e) / 2
            result.append(min(api_spans, key=lambda x: abs((x[0] + x[1]) / 2 - mid))[2])
    return result

In [15]:
# --- APIModel ------------------------------------------------------------
#
# Satisfies the interface ContextCiter uses:
#
#   model.device                               → torch.device("cpu")
#   model.generate(input_ids, ...)             → LongTensor [1, full_seq_len]
#   model(input_ids, attention_mask, labels)   → obj with .logits [bs, seq, V]
#
# generate():
#   We already have the RESPONSE from the backend API, so we skip a live
#   generation call. We append encoded RESPONSE tokens to the prompt.
#
# __call__():
#   For each ablated batch item:
#   1. Decode prompt / response from input_ids / labels via GPT-2.
#   2. Parse the ChatML prompt to extract user_content (masked context + query).
#   3. Call the LLM endpoint with prompt_logprobs=1 to score the response.
#   4. Align API token logprobs onto GPT-2 token boundaries.
#   5. Build fake logits so _compute_logit_probs returns the aligned logprobs:
#
#      logits[b, pos, label_id] = api_logprob + log(V-1),  all others = 0
#      → softmax-based loss recovers api_logprob exactly.

class APIModel:

    device = torch.device("cpu")

    def __init__(self, response: str):
        self._response = response

    def generate(self, input_ids: torch.Tensor, **kwargs) -> torch.Tensor:
        resp_ids = tokenizer.encode(self._response, add_special_tokens=False)
        full_ids = input_ids[0].tolist() + resp_ids
        return torch.tensor([full_ids], dtype=torch.long)

    def __call__(self, input_ids, attention_mask=None, labels=None, **kwargs):
        bs, seq_len = input_ids.shape
        V      = tokenizer.vocab_size
        logits = torch.zeros(bs, seq_len, V)
        log_V1 = float(np.log(V - 1))

        for b in range(bs):
            ids = input_ids[b].tolist()
            lbs = labels[b].tolist()

            resp_start   = next(i for i, l in enumerate(lbs) if l != -100)
            prompt_ids   = ids[:resp_start]
            response_ids = lbs[resp_start:]

            prompt_text   = tokenizer.decode(prompt_ids)
            response_text = tokenizer.decode(response_ids)
            user_content  = _extract_user_content(prompt_text)

            api_tokens = _api_response_token_logprobs(user_content, response_text)
            aligned    = _align_to_gpt2_tokens(api_tokens, response_text, response_ids)

            for j, (rid, alp) in enumerate(zip(response_ids, aligned)):
                pos = resp_start - 1 + j
                logits[b, pos, rid] = alp + log_V1

        return SimpleNamespace(logits=logits)

In [16]:
# --- Wire up ContextCiter ------------------------------------------------
#
# CONTEXT is the joined text of all retrieved chunks. ContextCite partitions
# it into sentences (sources), then for each of the num_ablations masks it
# calls APIModel.__call__() which hits the LLM endpoint with prompt_logprobs=1
# to score RESPONSE given the masked context. Lasso regression then fits
# source weights to explain those score deltas.
#
# All chunks are passed in — filtering by retrieval score before this step
# would defeat the purpose, since ContextCite exists to surface chunks that
# influenced the answer regardless of their retrieval rank.

CONTEXT = "\n\n".join(chunk["content"] for chunk in data["knowledge_sources"])

model = APIModel(response=RESPONSE)

cc = ContextCiter(
    model=model,
    tokenizer=tokenizer,
    context=CONTEXT,
    query=QUERY,
    num_ablations=32,
    ablation_keep_prob=0.5,
    batch_size=1,
)

print("Sources:")
for i, s in enumerate(cc.sources):
    print(f"  [{i}] {s[:80]}{'...' if len(s) > 80 else ''}")

Sources:
  [0] ### 3.1.2.
  [1] Quy định cách sử dụng
  [2] - Số ngày nghỉ tối đa Nhân viên có thể đăng ký nghỉ = Số ngày còn lại của Phép n...
  [3] - Quy định ứng trước phép không áp dụng cho Nhân viên đã thông báo nghỉ việc và ...
  [4] - Công ty khuyến khích Nhân viên có kế hoạch sử dụng phép năm hợp lý trong từng ...
  [5] Số phép năm chưa sử dụng hết được bảo lưu sang tháng hai (theo dõi tối đa 2 ngày...
  [6] Vượt quá thời điểm này, số ngày phép bảo lưu không sử dụng hết mặc nhiên sẽ bị h...
  [7] Các ngày phép bảo lưu không được chuyển đổi thành tiền ở bất cứ trường hợp nào.
  [8] - Các trường hợp nghỉ hưởng lương dài ngày theo Thỏa thuận tạm hoãn HĐLĐ sẽ khôn...
  [9] - Trường hợp Nhân viên nghỉ việc, số ngày còn lại của Phép năm hiện tại được hưở...
  [10] |  | Tổng lương cơ bản và mức thưởng V_Bonus |  |
  [11] | :---------------- | :---------------------------------- | :---------------- |
  [12] | Tất toán phép năm | tối đa của tháng trên HĐLĐ | x |
  [13] |  | Số ngày công

In [17]:
# .response triggers generate() — uses the backend RESPONSE, no extra API call
print("Response:", cc.response)

Response: Dựa trên các tài liệu hiện có, số ngày nghỉ phép năm cụ thể cho mỗi nhân viên VNG **không được nêu rõ** trong các đoạn trích đã tìm thấy.

Tuy nhiên, các tài liệu cung cấp các quy định liên quan đến việc sử dụng và bảo lưu phép năm:
*   Nhân viên có thể đăng ký nghỉ tối đa bằng số ngày phép năm còn lại cộng thêm tối đa **05 ngày** phép năm tạm ứng của năm hiện tại.
*   Số ngày phép năm chưa sử dụng hết có thể được bảo lưu sang năm tiếp theo (tối đa **05 ngày**) và phải được sử dụng không muộn hơn ngày **31/03** của năm sau.
*   Các ngày phép bảo lưu không sử dụng hết sau thời hạn trên sẽ bị hủy bỏ và không được chuyển đổi thành tiền.

Để biết chính xác số ngày phép năm được cấp cho từng nhân viên (thường phụ thuộc vào thâm niên), bạn cần tham khảo thêm các phần khác trong chính sách hoặc liên hệ bộ phận Nhân sự.


In [18]:
# get_attributions() runs num_ablations calls to the LLM endpoint (prompt_logprobs)
# then fits a Lasso to identify which context sentences drove the response.
cc.get_attributions(as_dataframe=True, verbose=True)

Attributed: Dựa trên các tài liệu hiện có, số ngày nghỉ phép năm cụ thể cho mỗi nhân viên VNG **không được nêu rõ** trong các đoạn trích đã tìm thấy.

Tuy nhiên, các tài liệu cung cấp các quy định liên quan đến việc sử dụng và bảo lưu phép năm:
*   Nhân viên có thể đăng ký nghỉ tối đa bằng số ngày phép năm còn lại cộng thêm tối đa **05 ngày** phép năm tạm ứng của năm hiện tại.
*   Số ngày phép năm chưa sử dụng hết có thể được bảo lưu sang năm tiếp theo (tối đa **05 ngày**) và phải được sử dụng không muộn hơn ngày **31/03** của năm sau.
*   Các ngày phép bảo lưu không sử dụng hết sau thời hạn trên sẽ bị hủy bỏ và không được chuyển đổi thành tiền.

Để biết chính xác số ngày phép năm được cấp cho từng nhân viên (thường phụ thuộc vào thâm niên), bạn cần tham khảo thêm các phần khác trong chính sách hoặc liên hệ bộ phận Nhân sự.


  0%|          | 0/32 [00:00<?, ?it/s]

USER CONTENT: Context: ### 3.1.2. Quy định cách sử dụng

- Số ngày nghỉ tối đa Nhân viên có thể đăng ký nghỉ = Số ngày còn lại của Phép năm thực tế tính đến thời điểm đăng ký + tối đa 05 ngày tạm ứng Phép năm của n
RESPONSE TEXT: Dựa trên các tài liệu hiện có, số ngày nghỉ phép năm cụ thể cho mỗi nhân viên VNG **không được nêu rõ** trong các đoạn trích đã tìm thấy.

Tuy nhiên, các tài liệu cung cấp các quy định liên quan đến v
STATUS: 500
BODY: {"error":{"message":"litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.. Received Model Group=local-model-mini\nAvailable Model Group Fallbacks=None","type":null,"param":null,"code":"500"}}


HTTPError: 500 Server Error: Internal Server Error for url: https://chat.zingplay.com/api/v1/chat/completions